# Lab 02 — Train and Measure a Real BPE Tokenizer

**Goal:** train a small BPE tokenizer, inspect its vocabulary and tokenization behavior, and compare token efficiency across languages.

Free-first: CPU or free Colab.

The central lesson is that tokenization is a model-design decision because it changes sequence length, embedding/output size, compute, memory, and multilingual coverage.


In [ ]:
!pip -q install -U tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

corpus=[
    "The model learns patterns from data.",
    "A good tokenizer represents common pieces efficiently.",
    "Language models predict the next token.",
    "بیمار پس از عمل جراحی به خانه بازگشت.",
    "هوش مصنوعی می تواند زبان های مختلف را پردازش کند.",
    "भारत एक विविध देश है और कई भाषाएँ यहाँ बोली जाती हैं।",
    "العربية لغة واسعة الانتشار وغنية المفردات."
]
print("documents:",len(corpus))


## 1. Predict first

Before running the tokenizer, predict which language will use the most tokens for a sentence of similar semantic length.

Also predict what happens when the vocabulary size is increased.


In [ ]:
tokenizer=Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer=Whitespace()
trainer=BpeTrainer(
    vocab_size=128,
    min_frequency=1,
    special_tokens=["[UNK]","[PAD]"]
)
tokenizer.train_from_iterator(corpus,trainer=trainer)
print("vocab size:",tokenizer.get_vocab_size())
print("sample vocab:",list(tokenizer.get_vocab().items())[:20])


In [ ]:
examples=[
    "The patient returned home after surgery.",
    "بیمار پس از عمل جراحی به خانه بازگشت.",
    "रोगी सर्जरी के बाद घर लौट आया।",
    "عاد المريض إلى المنزل بعد الجراحة."
]
for text in examples:
    enc=tokenizer.encode(text)
    print("\n",text)
    print("tokens:",enc.tokens)
    print("count:",len(enc.ids))


## 2. Measure token fertility

One useful engineering measure is token count per word-like unit. It is not a universal quality metric, but it makes the cost of representation differences visible.


In [ ]:
import re
for text in examples:
    enc=tokenizer.encode(text)
    word_like=max(1,len(text.split()))
    print({"text":text,"tokens":len(enc.ids),"words":word_like,"tokens_per_word":len(enc.ids)/word_like})


## 3. Vocabulary-size experiment

Train another tokenizer with a larger vocabulary. Predict whether sequence lengths must always decrease enough to justify the larger embedding/output matrix.


In [ ]:
tokenizer_big=Tokenizer(BPE(unk_token="[UNK]"))
tokenizer_big.pre_tokenizer=Whitespace()
trainer_big=BpeTrainer(vocab_size=256,min_frequency=1,special_tokens=["[UNK]","[PAD]"])
tokenizer_big.train_from_iterator(corpus,trainer=trainer_big)
for text in examples:
    a=len(tokenizer.encode(text).ids)
    b=len(tokenizer_big.encode(text).ids)
    print({"text":text,"small_vocab":a,"large_vocab":b})
print("small vocab:",tokenizer.get_vocab_size(),"large vocab:",tokenizer_big.get_vocab_size())


## 4. Critical-thinking answers

**Q:** Does a larger vocabulary automatically produce a better tokenizer?

**A:** No. A larger vocabulary can shorten sequences but increases token embedding/output parameters and can change multilingual coverage. Evaluate the tradeoff on representative data.

**Q:** Why is a tokenizer particularly important for a national/multilingual model?

**A:** If a language is fragmented into inefficient pieces, both training and inference can become more expensive and the model gets fewer semantic units per context window. Measure every target language separately.

**Q:** What should be frozen for a fair tokenizer comparison?

**A:** Use the same evaluation corpus, same measurement method, and clearly report preprocessing differences.


## 5. Research exercise

Choose three languages and a representative domain corpus. Train at least two tokenizer configurations and report:

- vocabulary size;
- tokens per byte/character/word;
- sequence length distribution;
- rare-script behavior;
- estimated training/inference cost impact;
- qualitative examples.

Use the result to propose a tokenizer configuration rather than declaring a universal winner.
